## 02a General Splits

Carga de los 7 datasets de splits generales del dashboard de equipo.


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path


In [ ]:
general_paths = [
    "../../../00_data/00c_final/2024-25/dashboards/team_dashboard_by_general_splits__dataset_0.parquet",
    "../../../00_data/00c_final/2024-25/dashboards/team_dashboard_by_general_splits__dataset_1.parquet",
    "../../../00_data/00c_final/2024-25/dashboards/team_dashboard_by_general_splits__dataset_2.parquet",
    "../../../00_data/00c_final/2024-25/dashboards/team_dashboard_by_general_splits__dataset_3.parquet",
    "../../../00_data/00c_final/2024-25/dashboards/team_dashboard_by_general_splits__dataset_4.parquet",
    "../../../00_data/00c_final/2024-25/dashboards/team_dashboard_by_general_splits__dataset_5.parquet",
    "../../../00_data/00c_final/2024-25/dashboards/team_dashboard_by_general_splits__dataset_6.parquet",
]

dfs_general = []
for p in general_paths:
    df = pd.read_parquet(p)
    print("Leído:", p, "→", df.shape)
    dfs_general.append(df)

df_general_splits = pd.concat(dfs_general, ignore_index=True)
print("Total combinado:", df_general_splits.shape)


# 1. Encabezado & Setup

Este cuaderno amplía el análisis de splits generales del dashboard para comprender drivers de rendimiento y preparar modelos base.
Se documenta el proceso completo con fecha automática y configuración reproducible.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print(f"Última ejecución: {datetime.now():%Y-%m-%d %H:%M:%S}")


In [ ]:
FIG_DIR = Path('figures/02a_general_splits')
REP_DIR = Path('reports/02a_general_splits')
for folder in (FIG_DIR, REP_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print(f'Carpetas listas: {FIG_DIR.resolve()} | {REP_DIR.resolve()}')


## 2. Normalización & Control de Calidad

Se trabaja sobre una copia para conservar los datos originales, homogeneizando nombres y asegurando tipos numéricos consistentes.
Los reportes de integridad permiten monitorear valores faltantes sin detener la ejecución.

In [ ]:
data = df_general_splits.copy()
data_upper = data.copy()
data_upper.columns = [str(col).upper() for col in data_upper.columns]
print(f'Dimensiones originales: {data.shape}; con columnas en mayúsculas: {data_upper.shape}')


In [ ]:
expected_cols = ['TEAM_ID', 'SEASON_YEAR', 'GP', 'W', 'L', 'W_PCT', 'PLUS_MINUS', 'GROUP_SET', 'GROUP_VALUE']
missing = [col for col in expected_cols if col not in data.columns]
if missing:
    print('Columnas clave faltantes:', missing)
else:
    print('Todas las columnas clave están presentes.')


In [ ]:
id_like = {col for col in data.columns if 'ID' in str(col).upper()}
string_like = set(data.select_dtypes(include=['object', 'string']).columns)
protected = id_like.union(string_like).union({'GROUP_SET', 'GROUP_VALUE', 'TEAM_ABBREVIATION'})
numeric_candidates = [col for col in data.columns if col not in protected]
converted = {}
for col in numeric_candidates:
    before_dtype = data[col].dtype
    data[col] = pd.to_numeric(data[col], errors='coerce')
    converted[col] = (str(before_dtype), str(data[col].dtype))
conversion_report = pd.DataFrame.from_dict(converted, orient='index', columns=['before', 'after'])
print(conversion_report.head())


In [ ]:
nan_report = (data.isna().sum().to_frame(name='nan_count')
              .assign(nan_pct=lambda df: df['nan_count'] / len(data) * 100)
              .sort_values('nan_count', ascending=False))
nan_report.to_csv(REP_DIR / 'nan_report.csv')
print('Reporte de NaN generado:', REP_DIR / 'nan_report.csv')
nan_report.head(10)


## 3. Resumen Global

Se resumen las métricas principales para comprender distribuciones generales y detectar outliers tempranos.
Incluye histogramas y rankings de eficiencia global por temporada y equipo.

In [ ]:
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [col for col in data.columns if col not in numeric_cols]
summary_numeric = data[numeric_cols].describe().T
summary_categorical = data[non_numeric_cols].describe(include='all').T if non_numeric_cols else pd.DataFrame()
summary_numeric.to_csv(REP_DIR / 'summary_numeric.csv')
summary_categorical.to_csv(REP_DIR / 'summary_categorical.csv')
print('Descriptivos numéricos guardados en summary_numeric.csv')
summary_numeric.head()


In [ ]:
for target in ['W_PCT', 'PLUS_MINUS']:
    if target in data.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(data[target].dropna(), bins=30, kde=True, color='#1f77b4')
        plt.title(f'Distribución de {target}')
        plt.xlabel(target)
        plt.ylabel('Frecuencia')
        plt.tight_layout()
        outfile = FIG_DIR / f'hist_{target.lower()}.png'
        plt.savefig(outfile)
        plt.close()
        print(f'Histograma guardado: {outfile}')
    else:
        print(f'No se encontró {target} para histogramas.')


In [ ]:
ranking_path_top = REP_DIR / 'ranking_wpct_top10.csv'
ranking_path_bottom = REP_DIR / 'ranking_wpct_bottom10.csv'
if {'TEAM_ID', 'SEASON_YEAR', 'W_PCT', 'GROUP_SET'}.issubset(data.columns):
    mask_overall = data['GROUP_SET'].astype(str).str.lower().eq('overall')
    if mask_overall.any():
        ranking = (data[mask_overall]
                   .groupby(['TEAM_ID', 'SEASON_YEAR'])['W_PCT']
                   .mean()
                   .reset_index()
                   .sort_values('W_PCT', ascending=False))
        ranking.head(10).to_csv(ranking_path_top, index=False)
        ranking.tail(10).to_csv(ranking_path_bottom, index=False)
        display(ranking.head(10))
        display(ranking.tail(10))
        print('Rankings guardados.')
    else:
        print('No hay registros Overall para generar ranking.')
else:
    print('Faltan columnas para generar ranking por TEAM_ID y SEASON_YEAR.')


## 4. Correlaciones Exhaustivas

Las métricas se agrupan por temática para interpretar su relación con W_PCT y PLUS_MINUS.
Se calculan correlaciones Pearson y Spearman, además de mapas de calor centrados en los objetivos.

In [ ]:
metric_blocks = {
    'volumen': ['FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'REB', 'OREB', 'DREB', 'AST', 'TOV', 'PTS'],
    'eficiencia': ['FG_PCT', 'FG3_PCT', 'FT_PCT', 'TS_PCT', 'EFG_PCT'],
    'creacion_perdidas': ['AST_RATIO', 'AST_TOV', 'AST_TO', 'PCT_AST'],
    'rebote': ['OREB_PCT', 'DREB_PCT', 'REB_PCT'],
    'defensa': ['STL', 'BLK', 'BLKA', 'PF', 'PF_PCT'],
    'resultado': ['W', 'L', 'W_PCT', 'PLUS_MINUS']
}
for block, cols in metric_blocks.items():
    metric_blocks[block] = [col for col in cols if col in data.columns]
print({k: len(v) for k, v in metric_blocks.items()})


In [ ]:
targets = [col for col in ['W_PCT', 'PLUS_MINUS'] if col in data.columns]
metrics = [col for col in numeric_cols if col not in {'W_PCT', 'PLUS_MINUS'}]
if targets and metrics:
    pearson = data[metrics + targets].corr(method='pearson')[targets].drop(targets)
    spearman = data[metrics + targets].corr(method='spearman')[targets].drop(targets)
    corr_table = pearson.join(spearman, lsuffix='_pearson', rsuffix='_spearman')
    for tgt in targets:
        corr_table[f'{tgt}_abs'] = corr_table[[f'{tgt}_pearson', f'{tgt}_spearman']].abs().max(axis=1)
    corr_table = corr_table.sort_values(by=[f'{targets[0]}_abs'], ascending=False)
    corr_table.to_csv(REP_DIR / 'corr_table_metrics_vs_targets.csv')
    display(corr_table.head(15))
else:
    print('No hay métricas numéricas o targets disponibles para correlaciones.')


In [ ]:
if targets and metrics:
    spearman_matrix = data[metrics + targets].corr(method='spearman')
    for tgt in targets:
        plt.figure(figsize=(12, 8))
        sns.heatmap(spearman_matrix[[tgt]].sort_values(by=tgt, ascending=False), annot=False, cmap='coolwarm', center=0)
        plt.title(f'Correlaciones Spearman con {tgt}')
        plt.tight_layout()
        outfile = FIG_DIR / f'heatmap_spearman_{tgt.lower()}.png'
        plt.savefig(outfile)
        plt.close()
        print(f'Heatmap guardado: {outfile}')
else:
    print('Sin targets para los mapas de calor.')


## 5. Segmentaciones Clave

Se exploran diferencias por tipo de split y otras dimensiones disponibles para revelar contextos con mayor impacto en el rendimiento.
Las tablas y gráficos permiten comparar fácilmente cada segmento.

In [ ]:
group_columns = []
for col in ['GROUP_SET', 'GROUP_VALUE', 'DATASET', 'SEASON_YEAR', 'SEASON_TYPE']:
    if col in data.columns:
        group_columns.append(col)
print('Columnas de segmentación disponibles:', group_columns)
segment_tables = {}
if group_columns:
    for col in group_columns:
        grouped = data.groupby(col)[numeric_cols].mean().reset_index()
        segment_tables[col] = grouped
        grouped.to_csv(REP_DIR / f'segment_mean_{col.lower()}.csv', index=False)
else:
    print('No hay columnas de segmentación disponibles.')


In [ ]:
if {'GROUP_SET', 'GROUP_VALUE', 'W_PCT'}.issubset(data.columns):
    for group_set, df_group in data.groupby('GROUP_SET'):
        chart_data = df_group.groupby('GROUP_VALUE')['W_PCT'].mean().sort_values(ascending=False)
        plt.figure(figsize=(10, 5))
        sns.barplot(x=chart_data.index, y=chart_data.values, palette='viridis')
        plt.title(f'W_PCT medio por GROUP_VALUE - {group_set}')
        plt.xlabel('GROUP_VALUE')
        plt.ylabel('W_PCT medio')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        outfile = FIG_DIR / f'bar_wpct_{group_set.lower().replace(" ", "_")}.png'
        plt.savefig(outfile)
        plt.close()
        print(f'Figura guardada: {outfile}')
else:
    print('Faltan columnas para generar gráficos por GROUP_SET/GROUP_VALUE.')


## 6. Feature Engineering (sin fuga)

Se crean ratios y métricas por 48 minutos que describen eficiencia sin introducir variables derivadas del resultado.
También se identifican columnas con posible fuga para excluirlas de los modelos.

In [ ]:
data_fe = data.copy()
ratio_pairs = {
    'AST_TOV_RATIO': ('AST', 'TOV'),
    'OREB_REB_RATIO': ('OREB', 'REB'),
    'DREB_REB_RATIO': ('DREB', 'REB'),
    'FGM_FGA_RATIO': ('FGM', 'FGA'),
    'FG3M_FG3A_RATIO': ('FG3M', 'FG3A'),
    'FTM_FTA_RATIO': ('FTM', 'FTA')
}
for new_col, (num, den) in ratio_pairs.items():
    if num in data_fe.columns and den in data_fe.columns:
        data_fe[new_col] = np.where(data_fe[den].abs() > 1e-6, data_fe[num] / data_fe[den], np.nan)
        print(f'Creado ratio {new_col}')
volume_cols = ['PTS', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA', 'OREB', 'DREB']
if 'MIN' in data_fe.columns:
    for col in volume_cols:
        if col in data_fe.columns:
            data_fe[f'{col}_PER48'] = np.where(data_fe['MIN'].abs() > 1e-6, data_fe[col] / data_fe['MIN'] * 48, np.nan)
else:
    print('No se encontró MIN para calcular métricas por 48 minutos.')
leakage_cols = [col for col in data_fe.columns if col.upper() in {'W', 'L'} or col.upper().endswith('_RANK') or col.upper() in {'W_PCT', 'PLUS_MINUS'}]
print('Columnas marcadas como fuga potencial:', leakage_cols[:10])


## 7. Drivers Ordenados

Los indicadores que mejor explican los targets se consolidan en una tabla priorizada para guiar el análisis.
Se exporta un ranking combinado con correlaciones Pearson y Spearman.

In [ ]:
if 'pearson' in locals() and 'spearman' in locals() and not pearson.empty and not spearman.empty:
    drivers = []
    for metric in pearson.index:
        row = {'metric': metric}
        for tgt in targets:
            row[f'{tgt}_pearson'] = pearson.loc[metric, tgt] if metric in pearson.index else np.nan
            row[f'{tgt}_spearman'] = spearman.loc[metric, tgt] if metric in spearman.index else np.nan
            row[f'{tgt}_score'] = max(abs(row[f'{tgt}_pearson']), abs(row[f'{tgt}_spearman']))
        drivers.append(row)
    drivers_df = pd.DataFrame(drivers)
    sort_cols = [f'{tgt}_score' for tgt in targets]
    drivers_df = drivers_df.sort_values(by=sort_cols, ascending=False)
    drivers_df.to_csv(REP_DIR / 'drivers_ranked.csv', index=False)
    display(drivers_df.head(20))
else:
    print('Sin targets definidos para drivers.')


## 8. Modelado Base — Regresión W_PCT

Se construyen modelos base para estimar el porcentaje de victorias respetando un esquema temporal.
Los resultados sirven como referencia para futuras iteraciones y mejoras.

In [ ]:
model_results_reg = []
if 'W_PCT' in data_fe.columns:
    model_data = data_fe.drop(columns=[col for col in data_fe.columns if col in leakage_cols], errors='ignore')
    model_data = model_data.dropna(subset=['W_PCT'])
    feature_cols = [col for col in model_data.select_dtypes(include=[np.number]).columns if col not in {'W_PCT'}]
    if feature_cols:
        if 'SEASON_YEAR' in data_fe.columns and data_fe['SEASON_YEAR'].notna().any():
            season_sorted = data_fe[['SEASON_YEAR']].dropna().sort_values('SEASON_YEAR')
            test_season = season_sorted.iloc[-1, 0]
            train_mask = data_fe['SEASON_YEAR'] != test_season
            test_mask = data_fe['SEASON_YEAR'] == test_season
            X_train = model_data.loc[train_mask, feature_cols]
            y_train = model_data.loc[train_mask, 'W_PCT']
            X_test = model_data.loc[test_mask, feature_cols]
            y_test = model_data.loc[test_mask, 'W_PCT']
            if X_test.empty:
                X_train, X_test, y_train, y_test = train_test_split(model_data[feature_cols], model_data['W_PCT'], test_size=0.25, random_state=42)
        else:
            X_train, X_test, y_train, y_test = train_test_split(model_data[feature_cols], model_data['W_PCT'], test_size=0.25, random_state=42)
        models = {
            'Ridge': Pipeline([('scaler', StandardScaler()), ('model', Ridge(random_state=42))]),
            'RandomForest': RandomForestRegressor(random_state=42)
        }
        for name, model in models.items():
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            result = {
                'model': name,
                'MAE': mean_absolute_error(y_test, preds),
                'RMSE': mean_squared_error(y_test, preds, squared=False),
                'R2': r2_score(y_test, preds)
            }
            model_results_reg.append(result)
        reg_report = pd.DataFrame(model_results_reg)
        reg_report.to_csv(REP_DIR / 'regression_report_wpct.csv', index=False)
        display(reg_report)
    else:
        print('No hay columnas numéricas disponibles para el modelo de regresión.')
else:
    print('W_PCT no está disponible para el modelado.')


## 9. Modelado Proxy — Clasificación WIN_FLAG

Se crea una etiqueta binaria para estimar victorias manteniendo el mismo corte temporal de entrenamiento.
Los modelos base entregan métricas esenciales de clasificación.

In [ ]:
classification_results = []
conf_matrix = None
if 'W_PCT' in data_fe.columns:
    win_flag = (data_fe['W_PCT'] >= 0.5).astype(int)
    clf_data = data_fe.drop(columns=[col for col in data_fe.columns if col in leakage_cols], errors='ignore')
    clf_data = clf_data.assign(WIN_FLAG=win_flag)
    clf_data = clf_data.dropna(subset=['WIN_FLAG'])
    feature_cols = [col for col in clf_data.select_dtypes(include=[np.number]).columns if col not in {'W_PCT', 'WIN_FLAG'}]
    if feature_cols:
        if 'SEASON_YEAR' in data_fe.columns and data_fe['SEASON_YEAR'].notna().any():
            season_sorted = data_fe[['SEASON_YEAR']].dropna().sort_values('SEASON_YEAR')
            test_season = season_sorted.iloc[-1, 0]
            train_mask = data_fe['SEASON_YEAR'] != test_season
            test_mask = data_fe['SEASON_YEAR'] == test_season
            X_train = clf_data.loc[train_mask, feature_cols]
            y_train = win_flag.loc[train_mask]
            X_test = clf_data.loc[test_mask, feature_cols]
            y_test = win_flag.loc[test_mask]
            if X_test.empty:
                X_train, X_test, y_train, y_test = train_test_split(clf_data[feature_cols], win_flag, test_size=0.25, random_state=42, stratify=win_flag)
        else:
            X_train, X_test, y_train, y_test = train_test_split(clf_data[feature_cols], win_flag, test_size=0.25, random_state=42, stratify=win_flag)
        classifiers = {
            'LogisticRegression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
            'RandomForestClassifier': RandomForestClassifier(random_state=42, class_weight='balanced')
        }
        for name, clf in classifiers.items():
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)
            probas = clf.predict_proba(X_test)[:, 1] if hasattr(clf, 'predict_proba') else preds
            result = {
                'model': name,
                'accuracy': accuracy_score(y_test, preds),
                'balanced_accuracy': balanced_accuracy_score(y_test, preds),
                'precision': precision_score(y_test, preds, zero_division=0),
                'recall': recall_score(y_test, preds, zero_division=0),
                'f1': f1_score(y_test, preds, zero_division=0),
                'roc_auc': roc_auc_score(y_test, probas) if len(np.unique(y_test)) > 1 else np.nan
            }
            classification_results.append(result)
            if name == 'RandomForestClassifier':
                conf_matrix = confusion_matrix(y_test, preds)
        clf_report = pd.DataFrame(classification_results)
        clf_report.to_csv(REP_DIR / 'classification_report_winflag.csv', index=False)
        display(clf_report)
        if conf_matrix is not None:
            plt.figure(figsize=(4, 4))
            sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                        xticklabels=['Loss', 'Win'], yticklabels=['Loss', 'Win'])
            plt.title('Matriz de confusión - RandomForestClassifier')
            plt.xlabel('Predicción')
            plt.ylabel('Real')
            plt.tight_layout()
            outfile = FIG_DIR / 'confusion_matrix_winflag.png'
            plt.savefig(outfile)
            plt.close()
            print(f'Matriz de confusión guardada en {outfile}')
    else:
        print('No hay features numéricos válidos para clasificación.')
else:
    print('W_PCT no disponible; no se genera WIN_FLAG.')


## 10. Importancias / Interpretabilidad

Se reportan las variables más influyentes del RandomForest de regresión para respaldar recomendaciones accionables.
Se exporta el top 20 y se visualiza en un gráfico ordenado.

In [ ]:
if model_results_reg and any(res['model'] == 'RandomForest' for res in model_results_reg):
    rf = RandomForestRegressor(random_state=42)
    rf.fit(model_data[feature_cols], model_data['W_PCT'])
    importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
    top_importances = importances.head(20)
    importance_df = importances.to_frame(name='importance').reset_index().rename(columns={'index': 'feature'})
    importance_df.to_csv(REP_DIR / 'feature_importances_rf.csv', index=False)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_importances.values, y=top_importances.index, palette='crest')
    plt.title('Importancias RandomForestRegressor - Top 20')
    plt.xlabel('Importancia')
    plt.ylabel('Feature')
    plt.tight_layout()
    outfile = FIG_DIR / 'feature_importances_rf_top20.png'
    plt.savefig(outfile)
    plt.close()
    print(f'Importancias guardadas en {outfile}')
else:
    print('No se dispone de modelo RandomForest entrenado para extraer importancias.')


## 11. Checklist para Escalar a Estimador W/L por Partido

Se delinean requisitos de datos y pasos metodológicos para evolucionar hacia predicciones a nivel juego.
El enfoque evita fugas al apoyarse solo en información disponible antes de cada partido.

### Consideraciones clave

- Requiere dataset game-level con GAME_ID, fechas, condición local/visitante, rival y métricas previas al encuentro.
- Features previas al partido: medias móviles (3/5/10), tendencias de diferencias, días de descanso, back-to-back y efecto localía.
- Calcular el espejo del rival únicamente con datos disponibles antes del juego mediante ventanas rolling alineadas.
- Validación temporal estricta por fecha; evitar splits aleatorios que mezclen futuro con pasado.
- Métricas recomendadas: balanced_accuracy, ROC-AUC y F1 para evaluar el clasificador final.

### Plantilla de funciones

1. `build_pre_match_features(games_df, window=5)` → Calcula rolling features, descansos y fusiona el espejo del rival.
2. `make_temporal_split(games_df, date_col)` → Separa train/test respetando la cronología y permite múltiples cortes de validación.
3. `train_match_classifier(X, y)` → Entrena clasificadores (LogReg, Gradient Boosting, RF) con búsqueda ligera y evalúa métricas clave.

## 12. Export Final & Resumen

Se verifica la existencia de los archivos generados y se sintetizan hallazgos clave y próximos pasos.
Así se asegura trazabilidad antes de conectar con boxscores a nivel partido.

In [ ]:
exports = {
    'reports': list(REP_DIR.glob('*.csv')),
    'figures': list(FIG_DIR.glob('*.png'))
}
for kind, files in exports.items():
    print(f'Archivos en {kind}:')
    for file in files:
        print(' -', file)
summary_points = []
if 'drivers_df' in locals():
    top_driver = drivers_df.head(1)['metric'].iloc[0] if not drivers_df.empty else 'N/A'
    summary_points.append(f'Driver principal identificado: {top_driver}.')
if 'reg_report' in locals():
    best_reg = reg_report.sort_values('RMSE').head(1)
    if not best_reg.empty:
        summary_points.append(f'Mejor modelo de regresión: {best_reg.iloc[0]["model"]} con RMSE={best_reg.iloc[0]["RMSE"]:.3f}.')
if 'clf_report' in locals():
    best_clf = clf_report.sort_values('roc_auc', ascending=False).head(1)
    if not best_clf.empty:
        summary_points.append(f'Mejor clasificador: {best_clf.iloc[0]["model"]} con ROC-AUC={best_clf.iloc[0]["roc_auc"]:.3f}.')
if not summary_points:
    summary_points.append('EDA ejecutado; revisar reportes para detalles.')
for point in summary_points:
    print('-', point)


### Conclusiones

- Los drivers destacados provienen de las correlaciones exportadas; priorizar su seguimiento en dashboards.
- Los modelos base entregan una línea de base inicial y necesitan optimización con features temporales adicionales.
- Próximo paso: integrar datos game-level para alimentar el estimador W/L por partido sin fuga de información.